In [1]:
from dotenv import load_dotenv
load_dotenv()
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.messages import SystemMessage ,HumanMessage,AIMessage
from langchain_tavily import TavilySearch

In [2]:
# 使用tavily作为web搜索工具
tavily = TavilySearch(
    max_results=5,
    topic="general"
)

@tool
def web_search(query: str):
    """Search the web for information"""
    return tavily.invoke(query)

In [3]:
from pydantic import BaseModel, Field

# Agent回答内容引用的网页信息
class Reference(BaseModel):
    title: str = Field(description="The title of the web page cited in the answer")
    url: str = Field(description="The url of the web page cited in the answer")

# Agent的回答内容
class AnswerInfo (BaseModel):
    answer: str = Field(description="The final answer for user")
    reference: list[Reference] = Field(description="The web pages cited in the answer")

# 创建智能体，使用预定义工具tavily
agent = create_agent(
    model="deepseek-chat",
    tools=[web_search],
    system_prompt="你是一个智能助手，你使用工具来解决用户问题。",
    response_format=AnswerInfo
)

# 调用agent
response = agent.invoke(
    {"messages": [HumanMessage(content="蒸蚌是什么梗？")]},
)

# 获取结构化输出
print(response['structured_response'])

answer='**「蒸蚌」是「真棒」的谐音梗**，源于近期火爆全网的"萝卜纸巾猫"热梗。\n\n## 梗的由来\n\n1. **起源**：抖音博主 @超级无敌大开门 养了一只名叫"大开门"的三花猫，主人教它辨认萝卜、纸巾等物品。这只猫其实根本分不清，全靠**察言观色、蒙答案**——先看看道具，再偷偷瞄一眼主人的表情，摸错了就立刻改。\n\n2. **"真棒"的出现**：每当猫咪碰巧蒙对答案时，主人就会用非常响亮、夸张（米老鼠式高音）的声音喊出**"真棒！"**，画面极其魔性洗脑。\n\n3. **"蒸蚌"的诞生**：视频被搬运到微博等平台后，网友把主人喊的"真棒"**空耳**（听错）成了"蒸蚌"，配上猫咪一脸懵的截图，形成了这个谐音梗。\n\n## 为什么火了\n\n- 🐱 **低门槛+强反差**：任何人都可以对自己的宠物或朋友喊一句"蒸蚌"，瞬间get"猫咪蒙题"的喜感。\n- 😂 **情绪感染力**：主人对笨猫无条件的夸赞和宠溺，这种"即使你分不清萝卜纸巾我也会夸你真棒"的温情打动了无数人。\n- 🎮 **全民参与**：从普通的萝卜纸巾认物游戏，演变成全网模仿潮——学生模拟期末考、医学生认医疗器械、职场人猜领导心思，甚至文旅、品牌、明星都被卷入这场"精神阳光普照"的狂欢。\n- 📺 **明星效应**：周深在直播中模仿喊"蒸蚌"等明星互动进一步放大了热度。\n\n## 含义\n\n目前"蒸蚌"没有其他特殊含义，就是**"真棒"的谐音，一种可爱、幽默的夸奖方式**。可以放心使用，对人对己都可以喊一声"蒸蚌"！🌟' reference=[Reference(title='周深蒸蚌#“蒸蚌”=“真棒”的谐音梗', url='https://www.sina.cn/news/detail/5247814261015270.html'), Reference(title='萝卜纸巾蒸蚌！全网小猫开始考研！', url='https://www.dealmoon.co.uk/guide/1107733'), Reference(title='全靠蒙的萝卜纸巾猫，咋就人传人了 - 虎嗅网', url='https://www.huxiu.com/article/4822539.html'), Reference(title='2025最后一个爆梗，“萝卜纸巾猫”究竟“真棒”在